In [ ]:
# ==================== ViT 模型训练（Jupyter 单单元格版本）====================
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import time
import gc
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.optim.lr_scheduler import ReduceLROnPlateau

import sys
sys.path.append('..')  # 添加上级目录到路径

try:
    from src.data.config import config, device
    from src.data.dataset import FundusDataset
    from src.data.class_balance import calculate_multilabel_weights, create_multilabel_balanced_sampler, WeightedBCEWithLogitsLoss
    from src.data.focal_loss import FocalLoss
    print("✅ 自定义模块导入成功！")
except Exception as e:
    print(f"❌ 自定义模块导入失败: {e}")
    raise

# ========== 主训练函数 ==========
def main():
    print("\n" + "="*60)
    print("ViT 模型训练开始")
    print("="*60 + "\n")
    
    # 1. 数据路径设置
    train_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Training Images'
    test_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Testing Images'
    val_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\validation images'
    excel_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\raw\ODIR-5K\data.xlsx'
    
    print("📁 数据路径设置完成")
    
    # 2. 加载数据集
    print("\n📊 加载数据集...")
    train_dataset = FundusDataset(train_dir, excel_dir, is_training=True)
    val_dataset = FundusDataset(val_dir, excel_dir, is_training=False)
    test_dataset = FundusDataset(test_dir, excel_dir, is_training=False)
    
    print(f"训练集大小: {len(train_dataset)}")
    print(f"验证集大小: {len(val_dataset)}")
    print(f"测试集大小: {len(test_dataset)}")
    
    # 3. 收集训练集标签用于类别平衡计算
    print("\n⚖️ 收集训练集标签...")
    all_train_labels = []
    for i in tqdm(range(len(train_dataset)), desc="收集标签"):
        _, labels, _ = train_dataset[i]
        all_train_labels.append(labels.numpy())
        if i % 1000 == 0:
            gc.collect()
    
    all_train_labels = np.stack(all_train_labels)
    print(f"标签数组形状: {all_train_labels.shape}")
    
    # 计算类别权重
    class_weights = calculate_multilabel_weights(
        all_train_labels,
        beta=0.999,
        clip_min=0.5,
        clip_max=5.0
    )
    print(f"\n类别权重:")
    for i, w in enumerate(class_weights):
        print(f"  类别 {i}: {w:.3f}")
    
    class_weights = class_weights.to(device)
    
    # 创建平衡采样器
    balanced_sampler = create_multilabel_balanced_sampler(
        all_train_labels,
        num_classes=config['num_classes'],
        replacement=True
    )
    
    # 4. 创建 DataLoader
    print("\n🔄 创建 DataLoader...")
    batch_size = config.get('batch_size', 16)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=balanced_sampler,
        num_workers=0,
        pin_memory=False,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    print(f"训练批次: {len(train_loader)}")
    print(f"验证批次: {len(val_loader)}")
    
    # 5. 加载预训练模型
    print("\n🤖 加载预训练模型...")
    local_model_path = config['pretrained_path']
    
    # 加载预训练的处理器
    processor = ViTImageProcessor.from_pretrained(local_model_path)
    print("✅ 处理器加载成功")
    
    # 加载预训练的ViT模型
    model = ViTForImageClassification.from_pretrained(
        local_model_path,
        num_labels=config['num_classes'],
        ignore_mismatched_sizes=True
    )
    model = model.to(device)
    print(f"✅ 模型加载成功！参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    
    # 6. 设置损失函数和优化器
    print("\n⚙️ 配置损失函数和优化器...")
    
    # 选择使用Focal Loss还是Weighted BCE
    use_focal_loss = True  # 设置为True使用Focal Loss
    
    if use_focal_loss:
        # 使用Focal Loss
        criterion = FocalLoss(
            alpha=class_weights,
            gamma=2.0,
            reduction='mean'
        )
        print("✅ 使用 Focal Loss，gamma=2.0")
    else:
        # 使用计算出的类别权重
        pos_rates = [0.06, 0.08, 0.02, 0.04, 0.04, 0.10, 0.06, 0.72]
        pos_weight = torch.tensor([1.0 / (r + 0.01) for r in pos_rates]).to(device)
        criterion = WeightedBCEWithLogitsLoss(
            class_weights=class_weights,
            reduction='mean'
        )
        print("✅ 使用 WeightedBCEWithLogitsLoss")
    
    # 优化器
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.get('learning_rate', 2e-5),
        weight_decay=config.get('weight_decay', 0.01)
    )
    
    # 学习率调度器
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=2,
        verbose=True,
        min_lr=1e-7
    )
    
    # 7. 初始化训练变量
    best_val_f1 = 0
    patience_counter = 0
    early_stop_patience = 5
    epochs = config.get('epochs', 15)
    
    # 创建保存目录
    os.makedirs(config['save_dir'], exist_ok=True)
    
    print(f"\n🚀 开始训练，共 {epochs} 轮")
    print("="*60)
    
    train_start_time = time.time()
    
    # 8. 训练循环
    for epoch in range(epochs):
        # ========== 训练阶段 ==========
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        train_strict_correct = 0
        train_strict_total = 0
        
        train_process = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        
        for image, labels, img_name in train_process:
            image = image.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(image)
            loss = criterion(outputs.logits, labels)
            
            probability = torch.sigmoid(outputs.logits)
            predict = (probability > 0.5).float()
            
            # 严格准确率（8个标签全对）
            strict_correct = (predict == labels).all(dim=1).sum().item()
            strict_total = labels.size(0)
            strict_acc = strict_correct / strict_total if strict_total > 0 else 0
            
            # 宽松准确率
            total_correct = (predict == labels).sum().item()
            total_elements = labels.numel()
            batch_acc = total_correct / total_elements if total_elements > 0 else 0
            
            loss.backward()
            optimizer.step()
            
            # 累计
            train_loss += loss.item()
            train_strict_correct += strict_correct
            train_strict_total += strict_total
            train_correct += total_correct
            train_total += total_elements
            
            train_process.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{batch_acc:.3f}',
                'strict': f'{strict_acc:.3f}'
            })
        
        # ========== 计算epoch平均值 ==========
        train_strict_accuracy = train_strict_correct / train_strict_total if train_strict_total > 0 else 0
        train_label_accuracy = train_correct / train_total if train_total > 0 else 0
        avg_train_loss = train_loss / len(train_loader)
        
        print(f"\nEpoch {epoch + 1}:")
        print(f"  损失: {avg_train_loss:.4f}")
        print(f"  标签准确率: {train_label_accuracy:.3f} (每个标签)")
        print(f"  严格准确率: {train_strict_accuracy:.3f} (8标签全对)")
        
        # ========== 验证阶段 ==========
        model.eval()
        all_val_probs = []
        all_val_labels = []
        
        with torch.no_grad():
            for val_images, val_labels, _ in tqdm(val_loader, desc=f'Epoch {epoch+1} [Val]'):
                val_images = val_images.to(device)
                val_outputs = model(val_images)
                val_probs = torch.sigmoid(val_outputs.logits).cpu().numpy()
                
                all_val_probs.append(val_probs)
                all_val_labels.append(val_labels.numpy())
        
        if all_val_probs:
            all_val_probs = np.vstack(all_val_probs)
            all_val_labels = np.vstack(all_val_labels)
            
            # 计算验证集F1
            val_preds = (all_val_probs > 0.5).astype(int)
            val_f1 = f1_score(all_val_labels, val_preds, average='macro', zero_division=0)
            
            # 计算每个类别的F1
            per_class_f1 = f1_score(all_val_labels, val_preds, average=None, zero_division=0)
            print(f"  各类别F1: {[f'{f1:.3f}' for f1 in per_class_f1]}")
            
            # 计算少数类（2-6）的平均F1
            minority_f1 = np.mean(per_class_f1[2:7])
            print(f"  少数类平均F1: {minority_f1:.4f}")
            print(f"  验证集Macro F1: {val_f1:.4f}")
            
            # 用验证集F1保存模型
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                patience_counter = 0
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': avg_train_loss,
                    'strict_accuracy': train_strict_accuracy,
                    'label_accuracy': train_label_accuracy,
                    'val_f1': val_f1,
                    'per_class_f1': per_class_f1,
                    'config': config
                }, os.path.join(config['save_dir'], 'best_model_by_val_f1.pth'))
                print(f"  ✅ 保存验证集最佳模型！F1={val_f1:.4f}")
            else:
                patience_counter += 1
            
            # 学习率调度
            scheduler.step(val_f1)
        
        # 保存最新模型
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
            'accuracy': train_strict_accuracy,
            'config': config
        }, os.path.join(config['save_dir'], 'latest_model.pth'))
        
        # 清理内存
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        print("-" * 50)
        
        # 早停检查
        if patience_counter >= early_stop_patience:
            print(f"\n⏹️ 早停: {early_stop_patience}个epoch F1未提升")
            break
    
    # 9. 训练总结
    train_time = time.time() - train_start_time
    print("\n" + "="*60)
    print("🏁 训练完成！")
    print("="*60)
    print(f"总训练时间: {train_time / 60:.2f} 分钟")
    print(f"最佳验证集F1: {best_val_f1:.4f}")
    print(f"模型保存位置: {config['save_dir']}")
    
    return model, best_val_f1

# ========== 在 Jupyter 中直接运行 ==========
print("开始训练...")
model, best_f1 = main()
print(f"训练结束，最佳F1: {best_f1:.4f}")

✅ 自定义模块导入成功！
开始训练...

ViT 模型训练开始

📁 数据路径设置完成

📊 加载数据集...
训练集大小: 4906
验证集大小: 1046
测试集大小: 1048

⚖️ 收集训练集标签...


收集标签: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4906/4906 [00:30<00:00, 163.14it/s]


标签数组形状: (4906, 8)

📊 多标签类别统计:
  类别 0: 正样本=1596, 负样本=3310, 正比例=0.325
  类别 1: 正样本=1584, 负样本=3322, 正比例=0.323
  类别 2: 正样本=304, 负样本=4602, 正比例=0.062
  类别 3: 正样本=296, 负样本=4610, 正比例=0.060
  类别 4: 正样本=232, 负样本=4674, 正比例=0.047
  类别 5: 正样本=146, 负样本=4760, 正比例=0.030
  类别 6: 正样本=246, 负样本=4660, 正比例=0.050
  类别 7: 正样本=1374, 负样本=3532, 正比例=0.280

📊 类别权重:
  类别 0: 0.500
  类别 1: 0.500
  类别 2: 1.077
  类别 3: 1.102
  类别 4: 1.363
  类别 5: 2.078
  类别 6: 1.294
  类别 7: 0.500

类别权重:
  类别 0: 0.500
  类别 1: 0.500
  类别 2: 1.077
  类别 3: 1.102
  类别 4: 1.363
  类别 5: 2.078
  类别 6: 1.294
  类别 7: 0.500


Some weights of ViTForImageClassification were not initialized from the model checkpoint at C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🔄 创建 DataLoader...
训练批次: 306
验证批次: 66

🤖 加载预训练模型...
✅ 处理器加载成功


...

Epoch 3/20 [Train]: 100%|██████████████████████████████████████████████████████████████████████████████████| 306/306 [29:26<00:00,  5.77s/it, loss=0.0456, acc=0.898, strict=0.375]



Epoch 3:
  损失: 0.0491
  标签准确率: 0.900 (每个标签)
  严格准确率: 0.405 (8标签全对)


Epoch 3 [Val]: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66/66 [01:02<00:00,  1.05it/s]


  各类别F1: ['0.101', '0.349', '0.262', '0.726', '0.438', '0.298', '0.734', '0.112']
  少数类平均F1: 0.4914
  验证集Macro F1: 0.3775
--------------------------------------------------


Epoch 4/20 [Train]:   1%|▊                                                                                   | 3/306 [00:17<29:06,  5.76s/it, loss=0.0319, acc=0.922, strict=0.500]

In [6]:
"""
无需命令行的类别分布分析脚本
直接在 IPython/Jupyter 中运行
"""

import numpy as np

def check_class_distribution(y_train, y_val, class_names=None):
    """
    分析训练集和验证集的类别分布
    """
    # 判断标签格式
    if len(y_train.shape) > 1 and y_train.shape[1] > 1:
        # one-hot 编码
        train_dist = np.mean(y_train, axis=0)
        val_dist = np.mean(y_val, axis=0)
        train_counts = np.sum(y_train, axis=0)
        val_counts = np.sum(y_val, axis=0)
        n_classes = y_train.shape[1]
    else:
        # 整数编码
        n_classes = len(np.unique(np.concatenate([y_train, y_val])))
        train_dist = np.array([np.mean(y_train == i) for i in range(n_classes)])
        val_dist = np.array([np.mean(y_val == i) for i in range(n_classes)])
        train_counts = np.array([np.sum(y_train == i) for i in range(n_classes)])
        val_counts = np.array([np.sum(y_val == i) for i in range(n_classes)])
    
    if class_names is None:
        class_names = [f"类别{i}" for i in range(n_classes)]
    
    print("=" * 60)
    print("类别分布分析报告")
    print("=" * 60)
    
    print("\n📊 各类别在训练集中的比例:")
    print("-" * 40)
    for i, name in enumerate(class_names):
        print(f"  {name}: {train_dist[i]:.3f} ({int(train_counts[i])} 样本)")
    
    print("\n📊 各类别在验证集中的比例:")
    print("-" * 40)
    for i, name in enumerate(class_names):
        print(f"  {name}: {val_dist[i]:.3f} ({int(val_counts[i])} 样本)")
    
    # 计算分布差异
    diff = np.abs(train_dist - val_dist)
    print("\n📈 分布差异:")
    print("-" * 40)
    for i, name in enumerate(class_names):
        print(f"  {name}: {diff[i]:.3f}")
    
    # 统计信息
    print("\n📉 统计摘要:")
    print("-" * 40)
    print(f"  训练集总样本数: {int(np.sum(train_counts))}")
    print(f"  验证集总样本数: {int(np.sum(val_counts))}")
    print(f"  类别数量: {n_classes}")
    print(f"  平均分布差异: {np.mean(diff):.3f}")
    
    # 检查不平衡
    imbalance_ratio = np.max(train_counts) / np.min(train_counts)
    print(f"\n⚠️  不平衡比率: {imbalance_ratio:.2f}")
    if imbalance_ratio > 2:
        print("  警告: 数据集存在明显不平衡！")
    
    return {
        'train_dist': train_dist,
        'val_dist': val_dist,
        'diff': diff,
        'train_counts': train_counts,
        'val_counts': val_counts,
        'imbalance_ratio': imbalance_ratio
    }

def generate_sample_data():
    """生成示例数据"""
    np.random.seed(42)
    n_classes = 5
    n_samples = 1000
    
    # 生成不平衡数据
    train_probs = np.array([0.4, 0.25, 0.15, 0.12, 0.08])
    y_train = np.random.choice(n_classes, size=n_samples, p=train_probs)
    y_val = np.random.choice(n_classes, size=n_samples//5, p=train_probs)
    
    # 转换为 one-hot
    y_train_onehot = np.eye(n_classes)[y_train]
    y_val_onehot = np.eye(n_classes)[y_val]
    
    return y_train_onehot, y_val_onehot, ['A', 'B', 'C', 'D', 'E']

# 在 IPython/Jupyter 中直接运行这部分
if __name__ == "__main__":
    print("运行示例数据...")
    y_train, y_val, class_names = generate_sample_data()
    check_class_distribution(y_train, y_val, class_names)

运行示例数据...
类别分布分析报告

📊 各类别在训练集中的比例:
----------------------------------------
  A: 0.421 (421 样本)
  B: 0.250 (250 样本)
  C: 0.130 (130 样本)
  D: 0.120 (120 样本)
  E: 0.079 (79 样本)

📊 各类别在验证集中的比例:
----------------------------------------
  A: 0.360 (72 样本)
  B: 0.205 (41 样本)
  C: 0.165 (33 样本)
  D: 0.150 (30 样本)
  E: 0.120 (24 样本)

📈 分布差异:
----------------------------------------
  A: 0.061
  B: 0.045
  C: 0.035
  D: 0.030
  E: 0.041

📉 统计摘要:
----------------------------------------
  训练集总样本数: 1000
  验证集总样本数: 200
  类别数量: 5
  平均分布差异: 0.042

⚠️  不平衡比率: 5.33
  警告: 数据集存在明显不平衡！
